In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/DE_Project_Pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg_de_project", "Catalog")
dbutils.widgets.text("data_source", "customers_non_aws", "Data Source")

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

print(catalog, data_source)

# base_path=f"s3://sportsbar-fmcg-de-project-viswadithya/{data_source}/*.csv"
# print(base_path)

In [0]:
# df=spark.read\
#     .format("csv")\
#         .option("inferschema", True)\
#             .option("header", True)\
#                 .load(base_path)\
#                     .withColumn("read_timestamp", f.current_timestamp())\
#                         .select("*", "_metadata.file_name", "_metadata.file_size")

# display(df.limit(10))

In [0]:
# df.printSchema()

In [0]:
# df.write\
#     .format("delta")\
#         .mode("overwrite")\
#             .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df_bronze=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
df_bronze=df_bronze.withColumn("ingestion_timestamp", F.current_timestamp())
display(df_bronze.count())

In [0]:
df_bronze.printSchema()


## Handling invalid records

In [0]:
df_invalid_products=df_bronze.filter(
        F.col("customer_id").isNull() | 
        F.col("customer_name").isNull() | 
        F.col("city").isNull()
    )

In [0]:
df_invalid_products=df_invalid_products.withColumn(
    "rejection_reason",
    F.when(F.col("customer_id").isNull(), "Missing customer id")
    .when(F.col("customer_name").isNull(), "Missing customer name")
    .when(F.col("city").isNull(), "Missing city")
)

display(df_invalid_products)


##Quarantine

In [0]:
df_invalid_products.write\
    .format("delta")\
        .mode("append")\
            .option("mergeSchema", True)\
                .saveAsTable(f"{catalog}.{silver_schema}.quarantine_{data_source}")

## Valid Records

In [0]:
df_valid_products=df_bronze.filter(
        (F.col("customer_id").isNotNull() &
        F.col("customer_name").isNotNull() &
        F.col("city").isNotNull())
    )

In [0]:
display(df_bronze)

In [0]:
Cdc_timestamp=F.date_sub(F.current_timestamp(), 2)
df_valid_products=df_valid_products.filter(F.col("ingestion_timestamp")>Cdc_timestamp)

display(df_valid_products)

In [0]:
print("Total Bronze records:", df_bronze.count())
print("Valid records:", df_valid_products.count())
print("Invalid records:", df_invalid_products.count())

In [0]:
w=Window.partitionBy(F.col("customer_id")).orderBy(F.col("updated_at").desc())
df_valid_products=df_valid_products.withColumn("rn",
                                               F.row_number().over(w)
                                               )

display(df_valid_products)                                                  

In [0]:
df_valid_products=df_valid_products.filter(F.col("rn")==1)
display(df_valid_products)

df_silver=df_valid_products

In [0]:
deleted_df=df_silver.filter(F.col("status")=="NON_ACTIVE")

display(deleted_df)

In [0]:
df_silver=df_silver.filter(
    F.col("status")=='ACTIVE'
)

In [0]:
df_duplicates=df_silver.groupBy("customer_id").count().filter(f.col("count")>1)
display(df_duplicates)

In [0]:
# print("count before removing duplicates:- ", df_bronze.count())
# df_silver=df_bronze.dropDuplicates(['customer_id'])
# print("count after removing duplicates:- ", df_silver.count())

In [0]:
display(
    df_silver.filter(
        F.col("customer_name")!=F.trim(F.col("customer_name"))
    )
)

In [0]:
df_silver=df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)
df_silver.display()

In [0]:
display(
    df_silver.filter(
        F.col("customer_name")!=F.trim(F.col("customer_name"))
    )
)

In [0]:
df_silver.select("city").distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed=[ 'Bengaluru', 'Hyderabad', 'New Delhi']

df_silver=df_silver.replace(city_mapping, subset=['city'])\
    .withColumn("city",
                F.when(F.col("city").isNull(), None)
                .when(F.col("city").isin(allowed), F.col("city"))
                .otherwise(None)
                )
    
df_silver.select("city").distinct().show()

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver=df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap(F.col("customer_name")))
)

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter(F.col("city").isNull()).show()

In [0]:
null_customer_names=['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix=spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()], 
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver=(df_silver.join(
    df_fix, "customer_id", "left"
).withColumn(
    "city",
    F.coalesce("city", "fixed_city")
)
.drop("fixed_city"))

In [0]:
df_silver.display()

In [0]:
df_silver=df_silver.withColumn(
    "customer_id",
    F.col("customer_id").cast('string')
)

df_silver.printSchema()

In [0]:
df_silver=df_silver.withColumn("customer", f.concat_ws("-", "customer_name", f.coalesce(f.col("city"), F.lit("unknown"))))\
.withColumn("market",F.lit("India"))\
.withColumn("platform", F.lit("Sports Bar"))\
.withColumn("channel", F.lit("Acquisition"))

df_silver.display()

In [0]:
display(df_silver.limit(1))

In [0]:
df_silver=df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel", "updated_at", "ingestion_timestamp")

In [0]:
display(df_silver.limit(1))

In [0]:
silver_table = f"{catalog}.{silver_schema}.{data_source}"

In [0]:
if not spark.catalog.tableExists(silver_table):
    empty_df = spark.createDataFrame(
        [],
        """
        customer_id STRING,
        customer_name STRING,
        city STRING,
        customer STRING,
        market STRING,
        platform STRING,
        channel STRING,
        updated_at TIMESTAMP,
        ingestion_timestamp TIMESTAMP,
        effective_start_date TIMESTAMP,
        effective_end_date TIMESTAMP,
        is_current BOOLEAN
        """
    )

    empty_df.write\
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", True)\
        .saveAsTable(silver_table)

In [0]:
silver_delta=DeltaTable.forName(
    spark,
    silver_table
)

current_active_df=(
        spark.table(silver_table)
        .filter(
            F.col("is_current")==True
        ).cache()
    )

In [0]:
deletions_to_process_df=deleted_df.alias("b").join(
    current_active_df.alias("s"),
    F.col("b.customer_id")==F.col("s.customer_id"),
    "inner"
).select(
    F.col("b.customer_id"),
    F.col("b.updated_at").alias("delete_timestamp")
)\
.dropDuplicates(["customer_id"])

In [0]:
current_active_df.count()

In [0]:
silver_delta=DeltaTable.forName(spark, silver_table)

silver_delta.alias("s").merge(
    deletions_to_process_df.alias("d"),
    """
        s.customer_id=d.customer_id 
            and
        s.is_current=True
    """
).whenMatchedUpdate(
    condition="d.delete_timestamp>s.updated_at",
    set={
        "is_current":"false",
        "effective_end_date": "d.delete_timestamp"
    }
).execute()

In [0]:
silver_table=f"{catalog}.{silver_schema}.{data_source}"
bronze_df=df_silver

window_spec=Window.partitionBy("customer_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())

bronze_df=bronze_df.withColumn("row_num", F.row_number().over(window_spec)).filter(F.col("row_num")==1).drop("row_num")


In [0]:
active_df=bronze_df.filter(
    F.upper(F.trim(F.col("status"))) == "ACTIVE"
)

deleted_df=bronze_df.filter(
    F.upper(F.trim(F.col("status"))) == "NON_ACTIVE"
)


In [0]:


silver_df=spark.table(silver_table).filter(F.col("is_current")==True)

joined_df=silver_df.alias("s").join(
    active_df.alias("b"),
    F.col("s.customer_id")==F.col("b.customer_id"),
    "inner"
)

changed_df=joined_df.filter(
    ~(
            (F.col("s.customer_id").eqNullSafe(F.col("b.customer_id")))
            & (F.col("s.city").eqNullSafe(F.col("b.city")))
            & (F.col("s.effective_start_date") < F.col("b.updated_at"))
    )
).select("b.*")

new_df=active_df.join(
    silver_df.select("customer_id").distinct(),
    on="customer_id",
    how="left_anti"
)

In [0]:

def prepare_new_version(df):
    return df\
            .withColumn(
                "effective_start_date",
                F.col("ingestion_timestamp").cast("timestamp")
            )\
            .withColumn(
                "effective_end_date",
                F.lit(None).cast("timestamp")
            )\
            .withColumn(
                "is_current", F.lit("True")
            )

changed_versions = prepare_new_version(changed_df)
new_versions = prepare_new_version(new_df)

expire_rows = changed_versions.withColumn(
    "merge_key", F.col("customer_id")
)

insert_changed_rows = changed_versions.withColumn(
    "merge_key",
    F.lit(None).cast(changed_versions.schema["customer_id"].dataType)
)
insert_new_rows = new_versions.withColumn(
    "merge_key",
    F.lit(None).cast(new_versions.schema["customer_id"].dataType)
)
staged_df = (
    expire_rows
    .unionByName(insert_changed_rows)
    .unionByName(insert_new_rows)
)


In [0]:





delta_silver= DeltaTable.forName(spark, silver_table)

delta_silver.alias("t").merge(
    staged_df.alias("s"),
    """
        t.customer_id = s.merge_key
        AND t.is_current = true
    """
).whenMatchedUpdate(
    condition="s.merge_key IS NOT NULL",
    set={
        "effective_end_date":"s.effective_start_date",
        "is_current":"false"
    }
).whenNotMatchedInsert(
    values={
        "customer_id":"s.customer_id",
        "customer_name":"s.customer_name",
        "city":"s.city",
        "customer":"s.customer",
        "market":"s.market",
        "platform":"s.platform",
        "channel":"s.channel",
        "market":"s.market",
        "platform":"s.platform",
        "channel":"s.channel",
        "updated_at":"s.updated_at",
        "ingestion_timestamp":"s.ingestion_timestamp",
        "is_current":"s.is_current",
        "effective_start_date":"s.effective_start_date",
        "effective_end_date":"s.effective_end_date"
    }
).execute()



In [0]:
# df_silver.write\
#     .mode("overwrite")\
#         .format("delta")\
#             .option("delta.enableChangeDataFeed", True)\
#                 .option("mergeSchema", True)\
#                     .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
# df_silver=spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")

# df_gold=df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
# df_gold.write\
#     .mode("overwrite")\
#         .format("delta")\
#             .option("delta.enableChangeDataFeed", True)\
#                 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
# delta_table=DeltaTable.forName(spark, "fmcg_de_project.gold.dim_customers")
# df_child_customers=spark.table("fmcg_de_project.gold.sb_dim_customers").select(
#     f.col("customer_id").alias("customer_code"),
#     "customer",
#     "market",
#     "platform",
#     "channel"
# )

In [0]:
# delta_table.alias("target").merge(
#     source=df_child_customers.alias("source"),
#     condition="target.customer_code=source.customer_code"
# ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()